In [1]:
import math
import random

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

SUPPORTED_BASES = [2, 4, 7, 8, 11, 13]


# --- 1. CLASSICAL MATH ---

def compute_period(a, N):
    T = 1
    while pow(a, T, N) != 1:
        T += 1
    return T


def continued_fraction_denominators(num, den, max_value):
    denominators = []
    q_prev_prev, q_prev = 1, 0

    while den != 0:
        a_i = num // den
        q_curr = a_i * q_prev + q_prev_prev
        if q_curr >= max_value:
            break
        if q_curr > 0:
            denominators.append(q_curr)
        q_prev_prev, q_prev = q_prev, q_curr
        num, den = den, num % den

    return denominators


# --- 2. QUANTUM CIRCUIT ---

def controlled_amod15(a, power):
    if a not in SUPPORTED_BASES:
        raise ValueError(f"a must be one of {SUPPORTED_BASES}")

    U = QuantumCircuit(4)
    for _ in range(power):
        if a in (2, 13):
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in (7, 8):
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in (11, 4):
            U.swap(1, 3); U.swap(0, 2)
        if a in (7, 11, 13):
            for q in range(4):
                U.x(q)

    return U.to_gate(label=f"{a}^{power} mod 15").control()


def build_shor_circuit(n_count, n_aux, a):
    qc = QuantumCircuit(n_count + n_aux, n_count)
    for q in range(n_count):
        qc.h(q)
    qc.x(n_count)

    aux_qubits = [i + n_count for i in range(n_aux)]
    for q in range(n_count):
        qc.append(controlled_amod15(a, 2 ** q), [q] + aux_qubits)

    qc.append(QFTGate(n_count).inverse(), range(n_count))
    qc.measure(range(n_count), range(n_count))
    return qc


# --- 3. POST-PROCESSING ---

def is_successful_measurement(k, Q, T_expected, a, N):
    if k == 0:
        return False
    for T in continued_fraction_denominators(k, Q, N):
        if T == T_expected and T % 2 == 0 and pow(a, T // 2, N) != N - 1:
            return True
    return False

def calculate_overhead_j(p_ideal, p_noisy):
    if p_noisy <= 0 or p_noisy >= 1 or p_ideal >= 1:
        return float('inf')

    numerator = math.log(1 - p_ideal)
    denominator = math.log(1 - p_noisy)

    return numerator / denominator

# --- 4. UTILITIES ---

def pick_valid_base(N, supported):
    while True:
        a = random.randint(2, N - 1)
        if math.gcd(a, N) > 1 or a not in supported:
            continue
        return a


def evaluate_success(counts, n_count, T_expected, a, N, shots):
    Q = 2 ** n_count
    successes = 0
    for bitstring, count in counts.items():
        k = int(bitstring, 2)
        ok = is_successful_measurement(k, Q, T_expected, a, N)
        print(f"k={k:2}  count={count:3}  check={ok}")
        if ok:
            successes += count
    return successes / shots


# --- 5. MAIN CORE ---

def main():
    N_COUNT, N_AUX, N_TARGET, SHOTS = 4, 4, 15, 1000

    a = pick_valid_base(N_TARGET, SUPPORTED_BASES)
    T = compute_period(a, N_TARGET)

    qc = build_shor_circuit(N_COUNT, N_AUX, a)
    simulator = AerSimulator()
    counts = simulator.run(transpile(qc, simulator), shots=SHOTS).result().get_counts()

    print(f"N = {N_TARGET}, a = {a}, period T = {T}")
    p_success = evaluate_success(counts, N_COUNT, T, a, N_TARGET, SHOTS)
    print(f"Success probability: {p_success * 100:.2f}%")


if __name__ == "__main__":
    main()

N = 15, a = 11, period T = 2
k= 8  count=509  check=True
k= 0  count=491  check=False
Success probability: 50.90%
